In [1]:
# Cell 1
# =============================================================================
# Reference & Baseline Model Artifacts Loading
# Based on: https://github.com/DrBaldri4n/Forschungsarbeit_p2p-ids-alert-correlation
# =============================================================================

import os
import joblib

# Directory where baseline model artifacts are stored
MODEL_DIR = "models"

# Load the pretrained Random Forest classifier, label encoder, and standard scaler
rf_final = joblib.load(os.path.join(MODEL_DIR, "rf_final.joblib"))
le = joblib.load(os.path.join(MODEL_DIR, "label_encoder.joblib"))
scaler = joblib.load(os.path.join(MODEL_DIR, "scaler.joblib"))

# Verify classes and model setup
print(f"Loaded model artifacts from: '{MODEL_DIR}'")
print(f"Baseline RF classes: {le.classes_}")

Loaded model artifacts from: 'models'
Baseline RF classes: ['Backdoor' 'none' 'syn-flood']


In [2]:
# Cell 2
# =============================================================================
# Feature Transformation & Generating RF Attack Probabilities for Nodes A–H
# =============================================================================

import os
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder, StandardScaler

# Node identifiers and directory configuration
NODES = ["A", "B", "C", "D", "E", "F", "G", "H"]
BASE_DIR = "dataset/normalized"
OUTPUT_DIR = os.path.join(BASE_DIR, "attack_prob")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Feature setup matching baseline model specifications
TARGET_COL = "Attack"
FEATURE_COLS = ["shunt_voltage", "bus_voltage_V", "current_mA", "power_mW", "State"]
NUMERIC_FEATURES = ["shunt_voltage", "bus_voltage_V", "current_mA", "power_mW"]
EXPECTED_FEATURE_ORDER = [
    "shunt_voltage", "bus_voltage_V", "current_mA", "power_mW",
    "State_idle", "State_charging"
]

# Load and combine all training files to fit preprocessors consistently
print("Loading combined train data to fit LabelEncoder and StandardScaler...")
train_files = [os.path.join(BASE_DIR, f"Node_{node}_train_normalized.csv") for node in NODES]
df_train_all = pd.concat([pd.read_csv(p) for p in train_files], ignore_index=True)

# Fit LabelEncoder on the target column
le = LabelEncoder()
y_train_all = le.fit_transform(df_train_all[TARGET_COL])

def prepare_features(df, scaler_instance=None, fit_scaler=False):
    """
    One-hot encodes 'State' and standardizes numerical sensor columns.
    """
    X_raw = df[FEATURE_COLS].copy()
    
    # One-hot encode the categorical 'State' column
    X_encoded = pd.get_dummies(X_raw, columns=["State"], drop_first=False)
    
    # Ensure both binary state columns exist
    for col in ["State_idle", "State_charging"]:
        if col not in X_encoded.columns:
            X_encoded[col] = 0
            
    # Enforce strict column order
    X_ordered = X_encoded[EXPECTED_FEATURE_ORDER].copy()
    
    # Standardize numeric columns
    if fit_scaler:
        scaler_instance = StandardScaler()
        X_ordered[NUMERIC_FEATURES] = scaler_instance.fit_transform(X_ordered[NUMERIC_FEATURES])
    else:
        X_ordered[NUMERIC_FEATURES] = scaler_instance.transform(X_ordered[NUMERIC_FEATURES])
        
    return X_ordered.to_numpy(), scaler_instance

# Fit the StandardScaler on the pooled training features
_, fitted_scaler = prepare_features(df_train_all, fit_scaler=True)
print(f"Fitted RF classes: {le.classes_}")
print("StandardScaler successfully fitted on combined training data.\n")

# Generate and save class probabilities for all nodes and splits
prob_col_names = [f"prob_{cls_name}" for cls_name in le.classes_]

for node in NODES:
    for split in ["train", "test"]:
        input_path = os.path.join(BASE_DIR, f"Node_{node}_{split}_normalized.csv")
        output_path = os.path.join(OUTPUT_DIR, f"Node_{node}_{split}_normalized.csv")
        
        df_node = pd.read_csv(input_path)
        
        # Prepare feature matrix for inference
        X_node_np, _ = prepare_features(df_node, scaler_instance=fitted_scaler, fit_scaler=False)
        
        # Predict class probabilities using baseline Random Forest
        probabilities = rf_final.predict_proba(X_node_np)
        
        # Append class probabilities as new columns
        for idx, cls_name in enumerate(le.classes_):
            df_node[f"prob_{cls_name}"] = probabilities[:, idx]
            
        # Save augmented CSV
        df_node.to_csv(output_path, index=False)
        print(f"Saved: {output_path} | Sample probs: {df_node[prob_col_names].iloc[0].values}")

print(f"\nDone. All probability-augmented files are stored in: {OUTPUT_DIR}")

Loading combined train data to fit LabelEncoder and StandardScaler...
Fitted RF classes: ['Backdoor' 'none' 'syn-flood']
StandardScaler successfully fitted on combined training data.

Saved: dataset/normalized/attack_prob/Node_A_train_normalized.csv | Sample probs: [0. 1. 0.]
Saved: dataset/normalized/attack_prob/Node_A_test_normalized.csv | Sample probs: [0.2 0.8 0. ]
Saved: dataset/normalized/attack_prob/Node_B_train_normalized.csv | Sample probs: [0.14 0.86 0.  ]
Saved: dataset/normalized/attack_prob/Node_B_test_normalized.csv | Sample probs: [0.04 0.96 0.  ]
Saved: dataset/normalized/attack_prob/Node_C_train_normalized.csv | Sample probs: [0.07 0.93 0.  ]
Saved: dataset/normalized/attack_prob/Node_C_test_normalized.csv | Sample probs: [0.06 0.94 0.  ]
Saved: dataset/normalized/attack_prob/Node_D_train_normalized.csv | Sample probs: [0.07 0.93 0.  ]
Saved: dataset/normalized/attack_prob/Node_D_test_normalized.csv | Sample probs: [0.4 0.6 0. ]
Saved: dataset/normalized/attack_prob/No

In [3]:
# Cell 3
# =============================================================================
# Load Augmented Data & Verify Synchronous Time Alignment Across All Nodes
# =============================================================================

import os
import pandas as pd

NODES = ["A", "B", "C", "D", "E", "F", "G", "H"]
BASE_DIR = "dataset/normalized/attack_prob"

# Dictionaries to hold train and test DataFrames per node
node_train = {}
node_test = {}

for node in NODES:
    train_path = os.path.join(BASE_DIR, f"Node_{node}_train_normalized.csv")
    test_path  = os.path.join(BASE_DIR, f"Node_{node}_test_normalized.csv")
    
    node_train[node] = pd.read_csv(train_path)
    node_test[node]  = pd.read_csv(test_path)
    
    print(f"Node {node} -> Train rows: {len(node_train[node])}, Test rows: {len(node_test[node])}")

# Verify that all node time series are aligned row-by-row
train_lengths = [len(node_train[n]) for n in NODES]
test_lengths  = [len(node_test[n])  for n in NODES]

assert len(set(train_lengths)) == 1, "Error: Training sets have mismatched row counts (misaligned time steps)!"
assert len(set(test_lengths))  == 1, "Error: Test sets have mismatched row counts (misaligned time steps)!"

n_train = train_lengths[0]
n_test  = test_lengths[0]

print(f"\nSanity Check Passed: All 8 nodes are time-aligned ({n_train} train rows, {n_test} test rows).")

Node A -> Train rows: 30732, Test rows: 13182
Node B -> Train rows: 30732, Test rows: 13182
Node C -> Train rows: 30732, Test rows: 13182
Node D -> Train rows: 30732, Test rows: 13182
Node E -> Train rows: 30732, Test rows: 13182
Node F -> Train rows: 30732, Test rows: 13182
Node G -> Train rows: 30732, Test rows: 13182
Node H -> Train rows: 30732, Test rows: 13182

Sanity Check Passed: All 8 nodes are time-aligned (30732 train rows, 13182 test rows).


In [4]:
# Cell 4
# =============================================================================
# Sigmoid Weighting Function for Collaborative Peer Reports
# =============================================================================

import numpy as np

# Sigmoid function parameters based on thesis correlation model
BASE_WEIGHT = 1.0
MAX_WEIGHT  = 3.0
K_SIGMOID   = 0.5   # Sigmoid curve steepness factor
N0_SIGMOID  = 1.0   # Sigmoid inflection midpoint
NUM_OTHER_NODES = 7 # Number of reporting peer nodes (8 nodes total - 1 target node)

def sigmoid_report_weight(attack_probability, number_of_reports=NUM_OTHER_NODES):
    """
    Computes a bounded sigmoid confidence weight based on a peer node's attack probability.
    Higher attack probabilities yield higher aggregation weights.
    """
    prob_clipped = float(np.clip(attack_probability, 0.0, 1.0))
    
    # Map probability to a count-equivalent scale [1, number_of_reports]
    n = 1.0 + (number_of_reports - 1.0) * prob_clipped
    
    # Calculate sigmoid components
    sig = 1.0 / (1.0 + np.exp(-K_SIGMOID * (n - N0_SIGMOID)))
    sig_anchor = 1.0 / (1.0 + np.exp(-K_SIGMOID * (1.0 - N0_SIGMOID)))
    
    # Scale within [BASE_WEIGHT, MAX_WEIGHT]
    weight = BASE_WEIGHT + (MAX_WEIGHT - BASE_WEIGHT) * (sig - sig_anchor)
    return float(np.clip(weight, BASE_WEIGHT, MAX_WEIGHT))

# Quick verification of the weighting curve
print(f"Weight for p=0.0: {sigmoid_report_weight(0.0):.4f}")
print(f"Weight for p=0.5: {sigmoid_report_weight(0.5):.4f}")
print(f"Weight for p=1.0: {sigmoid_report_weight(1.0):.4f}")

Weight for p=0.0: 1.0000
Weight for p=0.5: 1.6351
Weight for p=1.0: 1.9051


In [1]:
# Cell 4a
# =============================================================================
# Utility Theory Weighting Function for Collaborative Peer Reports
# =============================================================================

import numpy as np

# Utility theory parameters based on the Negative Exponential Utility (CARA) model
BASE_WEIGHT = 1.0
MAX_WEIGHT  = 3.0
K_UTILITY   = 0.1   # Risk-aversion / curvature constant
NUM_OTHER_NODES = 7 # Number of reporting peer nodes (8 nodes total - 1 target node)

def utility_report_weight(attack_probability, number_of_reports=NUM_OTHER_NODES, k=K_UTILITY):
    """
    Computes a concave utility weight based on a peer node's attack probability
    using the Negative Exponential Utility Function with diminishing returns (CARA):
    w(n) = W_base + (W_max - W_base) * (1 - exp(-k * (n - 1)))
    """
    prob_clipped = float(np.clip(attack_probability, 0.0, 1.0))
    
    # Map attack probability to a count-equivalent scale [1, number_of_reports]
    n = 1.0 + (number_of_reports - 1.0) * prob_clipped
    
    # Calculate negative exponential utility weight
    weight = BASE_WEIGHT + (MAX_WEIGHT - BASE_WEIGHT) * (1.0 - np.exp(-k * (n - 1.0)))
    
    return float(np.clip(weight, BASE_WEIGHT, MAX_WEIGHT))

# Quick verification of the utility weighting curve
print(f"Utility Weight for p=0.0: {utility_report_weight(0.0):.4f}")
print(f"Utility Weight for p=0.5: {utility_report_weight(0.5):.4f}")
print(f"Utility Weight for p=1.0: {utility_report_weight(1.0):.4f}")

Utility Weight for p=0.0: 1.0000
Utility Weight for p=0.5: 1.5184
Utility Weight for p=1.0: 1.9024


In [2]:
# Cell 5
# =============================================================================
# Calculate Collaborative Aggregated Context Features (Excluding Target Node)
# =============================================================================

import numpy as np
import pandas as pd

NODES = ["A", "B", "C", "D", "E", "F", "G", "H"]
NUMERIC_FEATURES = ["shunt_voltage", "bus_voltage_V", "current_mA", "power_mW"]
PROB_COLS = [f"prob_{cls_name}" for cls_name in le.classes_]

def build_augmented_with_aggregation(node_reports, nodes, numeric_features, prob_cols):
    """
    Computes weighted peer-aggregated features and attack probabilities for each node at each time step.
    For each target node X at row t, the aggregation strictly includes the other 7 peer nodes.
    """
    n_rows = len(node_reports[nodes[0]])
    augmented_dict = {}
    
    for target_node in nodes:
        df_target = node_reports[target_node].copy()
        
        # Initialize aggregate feature columns
        for feature in numeric_features:
            df_target[f"agg_{feature}"] = np.nan
            
        for prob_col in prob_cols:
            df_target[prob_col.replace("prob_", "agg_prob_")] = np.nan
            
        # Metadata columns
        df_target["agg_weight_sum"] = np.nan
        df_target["agg_num_contributors"] = np.nan
        
        # Row-by-row synchronous aggregation across time step t
        for t in range(n_rows):
            weights = []
            values_num = {feat: [] for feat in numeric_features}
            values_prob = {p_col: [] for p_col in prob_cols}
            
            for other_node in nodes:
                # Exclude the target node itself
                if other_node == target_node:
                    continue
                
                # Attack probability = 1.0 - prob_none (or normal class)
                prob_normal = node_reports[other_node].at[t, f"prob_{le.classes_[0]}"]
                attack_prob = 1.0 - prob_normal
                w = sigmoid_report_weight(attack_prob)
                
                row_other = node_reports[other_node].iloc[t]
                
                # Check for validity of numeric metrics
                valid_entry = True
                for feat in numeric_features:
                    val = row_other[feat]
                    if not pd.notna(val):
                        valid_entry = False
                        break
                    values_num[feat].append(val)
                    
                if not valid_entry:
                    continue
                    
                # Check for validity of peer probabilities
                for p_col in prob_cols:
                    val = row_other[p_col]
                    if not pd.notna(val):
                        valid_entry = False
                        break
                    values_prob[p_col].append(val)
                    
                if not valid_entry:
                    continue
                    
                weights.append(w)
            
            if len(weights) == 0:
                continue
                
            weights_arr = np.asarray(weights, dtype=float)
            weight_sum = weights_arr.sum()
            
            # Weighted average for numeric sensor features
            for feat in numeric_features:
                vals = np.asarray(values_num[feat], dtype=float)
                df_target.at[t, f"agg_{feat}"] = np.sum(weights_arr * vals) / weight_sum
                
            # Weighted average for class probabilities
            for p_col in prob_cols:
                vals = np.asarray(values_prob[p_col], dtype=float)
                agg_col_name = p_col.replace("prob_", "agg_prob_")
                df_target.at[t, agg_col_name] = np.sum(weights_arr * vals) / weight_sum
                
            # Store metadata
            df_target.at[t, "agg_weight_sum"] = weight_sum
            df_target.at[t, "agg_num_contributors"] = len(weights)
            
        augmented_dict[target_node] = df_target
        
    return augmented_dict

print("Computing aggregated collaborative features for training sets...")
aug_train = build_augmented_with_aggregation(node_train, NODES, NUMERIC_FEATURES, PROB_COLS)

print("Computing aggregated collaborative features for test sets...")
aug_test = build_augmented_with_aggregation(node_test, NODES, NUMERIC_FEATURES, PROB_COLS)

print("Aggregation complete.")
print(f"Sample columns generated for Node A: {aug_train['A'].columns.tolist()}")

NameError: name 'le' is not defined

In [ ]:
# Cell 6
# =============================================================================
# Serialize Augmented DataFrames to Disk (.pkl)
# =============================================================================

import os
import joblib

AUG_DIR = "dataset/normalized/augmented"
os.makedirs(AUG_DIR, exist_ok=True)

# Save pickled augmented DataFrames for each node
for node in NODES:
    train_out = os.path.join(AUG_DIR, f"Node_{node}_aug_train.pkl")
    test_out  = os.path.join(AUG_DIR, f"Node_{node}_aug_test.pkl")
    
    joblib.dump(aug_train[node], train_out)
    joblib.dump(aug_test[node], test_out)

print(f"All augmented DataFrames successfully saved to: '{AUG_DIR}'")

In [3]:
# Cell 7
# =============================================================================
# Pipeline Summary & Sanity Verification
# =============================================================================

import os
import joblib
import numpy as np
import pandas as pd

AUG_DIR = "dataset/normalized/augmented"
NODES = ["A", "B", "C", "D", "E", "F", "G", "H"]
NUMERIC_FEATURES = ["shunt_voltage", "bus_voltage_V", "current_mA", "power_mW"]

summary_records = []
all_valid = True

print("=== Collaborative Aggregation Pipeline Verification ===\n")

for node in NODES:
    train_path = os.path.join(AUG_DIR, f"Node_{node}_aug_train.pkl")
    test_path  = os.path.join(AUG_DIR, f"Node_{node}_aug_test.pkl")
    
    # Check physical file existence
    train_exists = os.path.exists(train_path)
    test_exists  = os.path.exists(test_path)
    
    if not (train_exists and test_exists):
        print(f"❌ Missing files for Node {node}!")
        all_valid = False
        continue
        
    df_node_train = joblib.load(train_path)
    df_node_test  = joblib.load(test_path)
    
    # Sanity Check: Ensure aggregated metrics are not identical to local metrics
    diff_detected = True
    for feat in NUMERIC_FEATURES:
        local_vals = df_node_train[feat].to_numpy()
        agg_vals   = df_node_train[f"agg_{feat}"].to_numpy()
        if np.allclose(local_vals, agg_vals, rtol=1e-6, atol=1e-6):
            print(f"⚠️ Warning: Node {node} local '{feat}' matches 'agg_{feat}' exactly (check peer masking).")
            diff_detected = False
            all_valid = False
            
    summary_records.append({
        "Node": node,
        "Train Shape": df_node_train.shape,
        "Test Shape": df_node_test.shape,
        "Avg Weight Sum": round(df_node_train["agg_weight_sum"].mean(), 4),
        "Avg Peers/Row": round(df_node_train["agg_num_contributors"].mean(), 1),
        "Aggregation Valid": "Yes" if diff_detected else "No"
    })

# Render summary table
df_summary = pd.DataFrame(summary_records)
print(df_summary.to_string(index=False))

if all_valid:
    print(f"\nAll 8 nodes successfully augmented and serialized to '{AUG_DIR}'.")
    print("Ready for multi-node context model training in the next notebook.")

=== Collaborative Aggregation Pipeline Verification ===

Node Train Shape  Test Shape  Avg Weight Sum  Avg Peers/Row Aggregation Valid
   A (30732, 19) (13182, 19)         12.6469            7.0               Yes
   B (30732, 19) (13182, 19)         12.6267            7.0               Yes
   C (30732, 19) (13182, 19)         12.6194            7.0               Yes
   D (30732, 19) (13182, 19)         12.5925            7.0               Yes
   E (30732, 19) (13182, 19)         12.5367            7.0               Yes
   F (30732, 19) (13182, 19)         12.5353            7.0               Yes
   G (30732, 19) (13182, 19)         12.5633            7.0               Yes
   H (30732, 19) (13182, 19)         12.5615            7.0               Yes

All 8 nodes successfully augmented and serialized to 'dataset/normalized/augmented'.
Ready for multi-node context model training in the next notebook.
